# 🗂️ Subset & Model Selection
**ISLP Chapter 6 · Pattern Recognition for the Rest of Us**

> When you have many predictors, which ones should you include in the model? Subset selection methods systematically compare model subsets. Information criteria (Cp, BIC, adjusted R²) give principled ways to choose model size without a test set.

### What you'll learn
- Best subset selection: exhaustive search
- Forward and backward stepwise selection: greedy approximations
- Cp, BIC, and adjusted R² as model selection criteria
- Practical comparison with cross-validation

### Dataset: Hitters (ISLP) — 19 baseball predictors
### Time: ~50 minutes

## 🎯 Part 1 — Why Variable Selection?

With p predictors, using all of them can overfit — especially when p is large relative to n. Including irrelevant predictors:
- Increases variance of estimates
- Reduces interpretability
- May worsen prediction on new data

We want the smallest model that explains the data well.

In [ ]:
# Forward stepwise selection (greedy — each step adds best remaining feature)
def forward_stepwise(X, y, feature_names, max_features=None):
    if max_features is None: max_features = X.shape[1]
    selected = []
    remaining = list(range(X.shape[1]))
    results = []
    lr = LinearRegression()
    for step in range(max_features):
        best_score, best_feat = -np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            cv = cross_val_score(lr, X[:, candidate], y, cv=5,
                                scoring='neg_mean_squared_error').mean()
            if cv > best_score:
                best_score, best_feat = cv, feat
        selected.append(best_feat)
        remaining.remove(best_feat)
        results.append({'n_features': step+1, 'features': selected.copy(),
                       'cv_mse': -best_score,
                       'last_added': feature_names[best_feat]})
    return pd.DataFrame(results)

print('Running forward stepwise selection...')
fwd_results = forward_stepwise(X, y, features, max_features=15)
print(fwd_results[['n_features','last_added','cv_mse']].to_string())
print(f'\nBest model size: {fwd_results["cv_mse"].idxmin()+1} features')

In [ ]:
# Plot: CV MSE vs number of features
fig, ax = plt.subplots(figsize=(10, 4))
best_n = fwd_results['cv_mse'].idxmin()
ax.plot(fwd_results['n_features'], fwd_results['cv_mse'], 'o-', color='#1e5fa8', lw=2, markersize=6)
ax.axvline(fwd_results.loc[best_n, 'n_features'], color='#e85d20', lw=1.5, ls='--',
          label=f'Best: {fwd_results.loc[best_n,"n_features"]} features (CV MSE={fwd_results.loc[best_n,"cv_mse"]:.4f})')
ax.set_xlabel('Number of features')
ax.set_ylabel('5-fold CV MSE (log salary)')
ax.set_title('Forward Stepwise Selection — Hitters Dataset')
ax.legend()
plt.tight_layout(); plt.show()
best_features = fwd_results.loc[best_n, 'features']
print(f'\nSelected features: {[features[i] for i in best_features]}')

In [ ]:
# Information criteria: AIC, BIC, adjusted R²
# (computed on full training data — no cross-validation needed)
import statsmodels.api as sm

n = len(y)
X_sm = sm.add_constant(X[:, :10])  # first 10 features for illustration
model_sm = sm.OLS(y, X_sm).fit()

print('Model selection criteria (statsmodels):')  
print(f'  AIC:  {model_sm.aic:.2f}')
print(f'  BIC:  {model_sm.bic:.2f}')
print(f'  R²:   {model_sm.rsquared:.4f}')
print(f'  Adj R²: {model_sm.rsquared_adj:.4f}')
print()
print('Building models of increasing size and comparing criteria...')
for k in [1, 3, 5, 8, 10, 12, 15]:
    if k <= len(features):
        Xk = sm.add_constant(X[:, :k])
        mk = sm.OLS(y, Xk).fit()
        print(f'  p={k:<3} AIC={mk.aic:.1f}  BIC={mk.bic:.1f}  AdjR²={mk.rsquared_adj:.4f}')

In [ ]:
# Exercise workspace
# Task 1: Implement backward stepwise selection (start with all features, remove worst one each step)
# YOUR CODE HERE

# Task 2: Compare forward stepwise vs Lasso (from regularization notebook) on Hitters
# Which selects fewer features? Which has lower CV MSE?
from sklearn.linear_model import LassoCV
# YOUR CODE HERE

# Task 3: What features does the best 5-feature model select? Interpret each one
best_5 = fwd_results.loc[fwd_results['n_features']==5, 'features'].values[0]
print('Top 5 features:', [features[i] for i in best_5])
# YOUR CODE HERE — fit and report coefficients

In [ ]:
# @title 📝 Quiz — Subset & Model Selection
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is best subset selection and why is it computationally expensive?
# @markdown **Q2:** What is the difference between forward and backward stepwise selection?
# @markdown **Q3:** What does BIC penalize compared to AIC?
# @markdown **Q4:** Why is adjusted R² better than R² for model comparison?
# @markdown **Q5:** When would you use stepwise selection vs Lasso for variable selection?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Subset & Model Selection"
# @title 🤖 AI Feedback — run this cell for instant grading
# @markdown **Enter your GitHub username below, then click ▶ Run**
# @markdown
# @markdown No API key needed — AI grading runs directly inside Colab for free.
# @markdown
# @markdown *(If AI grading is unavailable, you still get completion-based feedback)*

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ─────────────────────────────────────────────────────────────
import json, textwrap, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade_with_gemini(answers_dict, nb_name):
    """Call Gemini inside Colab — no API key required."""
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)

    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )

    prompt = f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

Student answers ({n_done}/{n_total} answered):
{qa_lines if qa_lines else "(none provided)"}

Grade conceptual understanding, not exact wording. Accept reasonable paraphrases.
Be encouraging and specific. Reply ONLY in this JSON format, no markdown:
{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions>",
  "tip": "<one specific thing to remember from this technique>"
}}"""

    # ── Attempt 1: Colab-native Gemini (zero config) ──────────
    try:
        import google.generativeai as genai
        import google.auth
        # Use Colab's ambient credentials — no API key needed
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        model  = genai.GenerativeModel("gemini-2.0-flash")
        result = model.generate_content(prompt)
        return result.text, "Gemini (Colab)"
    except Exception:
        pass

    # ── Attempt 2: Colab userdata key (if student added one) ──
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key and len(api_key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=api_key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini (API key)"
    except Exception:
        pass

    # ── Attempt 3: urllib fallback ─────────────────────────────
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key and len(api_key) > 10:
            import urllib.request
            url = (f"https://generativelanguage.googleapis.com/v1beta/"
                   f"models/gemini-2.0-flash:generateContent?key={api_key}")
            payload = json.dumps({
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
            }).encode()
            req = urllib.request.Request(url, data=payload,
                                          headers={"Content-Type":"application/json"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
                return data["candidates"][0]["content"]["parts"][0]["text"], "Gemini (urllib)"
    except Exception:
        pass

    return None, None

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "feedback":"No answers provided — fill in the quiz answers above and re-run.",
                "tip":"Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score":score,"grade":"Needs Review",
                "feedback":f"You answered {n_answered}/{n_total} questions but responses are very brief. Try to explain your reasoning in 1-2 sentences.",
                "tip":"Aim for a sentence or two per answer — quality over brevity."}
    elif n_answered == n_total:
        return {"quiz_score":score,"grade":"Good",
                "feedback":f"All {n_total} questions answered with good detail.",
                "tip":"Review any concepts that felt unclear and check the Further Reading links."}
    else:
        return {"quiz_score":score,"grade":"Needs Review",
                "feedback":f"{n_answered} of {n_total} questions answered. Complete the remaining {n_total-n_answered} for full credit.",
                "tip":"Answer all questions before submitting."}

def _print_result(result, username, nb_name, source):
    colors = {"Excellent":"\033[92m","Good":"\033[94m",
              "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R     = "\033[0m"
    grade = result.get("grade","See feedback")
    C     = colors.get(grade,"\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by  {source}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    for line in textwrap.wrap(result.get("feedback",""), width=56):
        print(f"  {line}")
    tip = result.get("tip","")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{len(answers)} provided")
    if username: print(f"  Student  : @{username}")

    raw, source = _grade_with_gemini(answers, _NB_TITLE)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            result = {"quiz_score":n_answered,
                      "grade":"Good" if n_answered==len(answers) else "Needs Review",
                      "feedback":raw[:400],"tip":""}
    else:
        print("  AI unavailable \u2014 using completion-based feedback\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE, source)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


In [ ]:
_NB_NAME_ = "Subset & Model Selection"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
